In [23]:
import pandas as pd
import numpy as np
import duckdb
import sklearn
import xgboost
import lightgbm
from unidecode import unidecode
import re
from rapidfuzz import process, fuzz

In [24]:
master_dataset = pd.read_parquet(
    "../data/processed_data/master_dataset.parquet"
)

##### Detectar variables constantes o prácticamente sin variabilidad

In [25]:
n_unique = master_dataset.nunique(dropna=False).sort_values()

n_unique.head(20)

changed_league         2
changed_team           2
transferred            2
season                 3
pen_miss               3
pen_post               3
direct_red_cards       3
own_goals              4
yellow_red_cards       4
pen_on_target          4
league_count           4
team_count             4
red_cards              4
crosses_not_claimed    5
pens_conceded          5
pens_saved             5
goals_freekick         6
pass_to_assist         6
saves_caught           7
pens_won               7
dtype: int64

##### Detectar columnas completamente vacías

In [26]:
all_null_cols = master_dataset.columns[
    master_dataset.isna().all()
].tolist()

all_null_cols

[]

##### Detectar columnas que son duplicados exactos de otras

In [27]:
duplicate_cols = []

columns = master_dataset.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        if master_dataset[columns[i]].equals(
            master_dataset[columns[j]]
        ):
            duplicate_cols.append(
                (columns[i], columns[j])
            )

duplicate_cols

[]

##### Selección de registros de jugadores under 23

Escogemos jugadores que tienen menos de 23 años, o que han cumplido 23 este mismo mes, por lo que el punto de corte es agosto del 2003.

In [28]:
df_under_23 = master_dataset[
    (master_dataset["date_of_birth"].dt.year >= 2003) & 
    (master_dataset["date_of_birth"].dt.year >= 8)
].copy()

In [31]:
df_under_23 = df_under_23[df_under_23["minutes"] > 0]

In [32]:
len(df_under_23)

2759

Seleccionamos los porteros. Para ello, se considera unicamente cualquier portero que haya hecho por lo menos 1 parada, o que este identificado como potero en la variable `pos` (contains("GK"))

In [ ]:
df_under_23_gk = df_under_23[
    (df_under_23["pos"].str.contains("GK", na=False)) | 
    (df_under_23["saves"] > 0)
]

len(df_under_23_gk)


61

Seleccionamos los jugadores de campo, descartando los que cumplen la condición utilizada para seleccionar los porteros, e incluyendo la condición de haber jugado por lo menos 150 minutos. 

Cualquier jugador que haya jugado menos de 150 minutos no cumple con suficiente tiempo de juego como para ser considerado. Se opta por una cantidad de minutos tan baja debido al tratarse de un análisis de jugadores jovenes, los cuales muchos de ellos no cuentan con tiempo regular de juego, pero que su rendimiento puede comenzar a ser análizado habiendo sumado al menos 150 minutos en total.

In [40]:
df_under_23_outfield = df_under_23[
    (df_under_23["minutes"] >= 150) &
    (df_under_23["pos"].str.contains("GK", na=False) == False) &
    (df_under_23["saves"] <= 0)
].copy()

len(df_under_23_outfield)

1839

In [30]:
for col in df_under_23.columns:
    print(col)

player
season
games
minutes
goals
assists
shots
key_passes
yellow_cards
red_cards
npg
xg
xag
npxg
xg_chain
xg_buildup
ninety_s
sofascore_rating_total
sofascore_rating_count
totw_appearances
starts
tackles
tackles_won
interceptions
clearances
blocked_shots
outfield_blocks
errors_leading_to_goal
errors_leading_to_shot
dribbled_past
aerials_won
aerials_lost
dribbles_completed
dribbles_attempted
ground_duels_won
duels_won
duels_lost
passes_total
passes_completed
passes_inaccurate
passes_final_third
passes_opp_half
passes_own_half
passes_opp_half_total
passes_own_half_total
long_balls_total
long_balls_completed
crosses_total
crosses_completed
chipped_passes_total
chipped_passes_completed
pass_to_assist
attempt_assists
big_chances_created
big_chances_missed
goals_inside_box
goals_outside_box
goals_headed
goals_left_foot
goals_right_foot
goals_penalty
goals_freekick
own_goals
goals_assists
hit_woodwork
shots_inside_box
shots_outside_box
shots_on_target
shots_off_target
shots_set_piece
touches

### Cálculo de ratios

##### Ratio ofensivo

Producción total

In [41]:
offensive_total_cols = [
    "goals",
    "assists",
    "shots",
    "big_chances_created",
    "xg",
    "xag",
    "npxg"
]

Producción por 90 min

In [42]:
offensive_per90_cols = [
    "goals_per90",
    "assists_per90",
    "shots_per90",
    "big_chances_created_per90",
    "xg_per90",
    "xag_per90",
    "npxg_per90"
]

Eficiencia y calidad de definición

In [44]:
offensive_pct_cols = [
    "goal_conversion_pct",
    "shots_inside_box_conversion_pct",
    "shots_outside_box_conversion_pct",
    "xg_overperformance"
]

In [45]:
for col in offensive_total_cols:
    df_under_23_outfield[f"{col}_score"] = (
        df_under_23_outfield[col]
        .rank(pct=True) * 99
    )

for col in offensive_per90_cols:
    df_under_23_outfield[f"{col}_score"] = (
        df_under_23_outfield[col]
        .rank(pct=True) * 99
    )

for col in offensive_pct_cols:
    df_under_23_outfield[f"{col}_score"] = (
        df_under_23_outfield[col]
        .rank(pct=True) * 99
    )

In [46]:
total_score_cols = [
    f"{col}_score"
    for col in offensive_total_cols
]

per90_score_cols = [
    f"{col}_score"
    for col in offensive_per90_cols
]

pct_score_cols = [
    f"{col}_score"
    for col in offensive_pct_cols
]

In [ ]:
df_under_23_outfield["offensive_total_score"] = (
    df_under_23_outfield[total_score_cols]
    .mean(axis=1)
)

df_under_23_outfield["offensive_per90_score"] = (
    df_under_23_outfield[per90_score_cols]
    .mean(axis=1)
)

df_under_23_outfield["offensive_efficiency_score"] = (
    df_under_23_outfield[pct_score_cols]
    .mean(axis=1)
)

In [48]:
df_under_23_outfield["offensive_rating"] = (
    df_under_23_outfield["offensive_total_score"] * 0.35
    + df_under_23_outfield["offensive_per90_score"] * 0.45
    + df_under_23_outfield["offensive_efficiency_score"] * 0.20
)

In [49]:
df_under_23_outfield[["player", "team", "offensive_total_score", "offensive_per90_score", "offensive_efficiency_score", "offensive_rating"]]

,player,team,offensive_total_score,offensive_per90_score,offensive_efficiency_score,offensive_rating
11,Jordan James,Birmingham City,64.073526,48.511769,89.787072,62.213444
24,Adam Wharton,Blackburn Rovers | Crystal Palace,80.388954,56.302261,59.943121,65.460775
25,Andrew Moran,Blackburn Rovers,75.047891,63.669774,37.544231,62.427006
27,Ben Chrisene,Blackburn Rovers,27.666628,27.347471,23.810489,26.751780
39,Yasin Ayari,Blackburn Rovers,60.408996,65.934631,39.831371,58.780006
...,...,...,...,...,...,...
12712,Álex Balde,Barcelona | FC Barcelona,62.727686,44.505010,24.274412,46.836827
12715,Javier Rodríguez,Celta Vigo,60.755069,33.546027,23.130448,40.986076
12716,Jones El Abdellaoui,Celta Vigo,64.600326,66.661384,47.704751,62.148687
12717,Manuel Fernández,Celta Vigo,23.644488,20.287579,32.012993,23.807580


OFFENSIVE RATING

- 35% Producción total

- 45% Producción por 90

- 20% Eficiencia